In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import pandas as pd

df = pd.read_csv("/content/Dataset_A_clean.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nClass distribution:")
print(df["Is Fake"].value_counts())

print("\nClass distribution (%):")
print(df["Is Fake"].value_counts(normalize=True) * 100)

Dataset shape: (14564, 2)

Columns:
['Tamil', 'Is Fake']

Class distribution:
Is Fake
0    11662
1     2902
Name: count, dtype: int64

Class distribution (%):
Is Fake
0    80.074155
1    19.925845
Name: proportion, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split

X = df["Tamil"].astype(str)
y = df["Is Fake"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training samples: 11651
Testing samples: 2913

Training class distribution:
Is Fake
0    9329
1    2322
Name: count, dtype: int64

Testing class distribution:
Is Fake
0    2333
1     580
Name: count, dtype: int64


In [ ]:
from transformers import BertTokenizer

MODEL_NAME = "bert-base-multilingual-cased"

print("Loading mBERT tokenizer...")

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")

Loading mBERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [ ]:
sample_text = X_train.iloc[0]

tokens = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=200
)

print("Input text:")
print(sample_text[:300])

print("\nInput IDs length:", len(tokens["input_ids"]))
print("Attention mask length:", len(tokens["attention_mask"]))

Input text:
 ஆசிய கோப்பை கிரிக்கெட் | 7 முறைவாகை சூடிய இந்தியா - யார்...   

Input IDs length: 200
Attention mask length: 200


In [ ]:
from torch.utils.data import Dataset
import torch

class TamilNewsDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=200):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )
        }


MAX_LENGTH = 200

train_dataset = TamilNewsDataset(
    X_train,
    y_train,
    tokenizer,
    MAX_LENGTH
)

test_dataset = TamilNewsDataset(
    X_test,
    y_test,
    tokenizer,
    MAX_LENGTH
)

print("Training dataset:", len(train_dataset))
print("Testing dataset:", len(test_dataset))

Training dataset: 11651
Testing dataset: 2913


In [ ]:
sample = train_dataset[0]

print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label:", sample["labels"])

Input IDs shape: torch.Size([200])
Attention mask shape: torch.Size([200])
Label: tensor(0)


In [ ]:
from transformers import BertForSequenceClassification
import torch

MODEL_NAME = "bert-base-multilingual-cased"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)
print("GPU:", torch.cuda.get_device_name(0))

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print("\nModel loaded successfully.")

Using device: cuda
GPU: Tesla T4


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model loaded successfully.


In [ ]:
!nvidia-smi

Tue Aug 25 07:32:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P0             28W /   70W |     839MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 1457
Testing batches: 365


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train.to_numpy()
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class weights:")
print(class_weights)

Class weights:
tensor([0.6245, 2.5088], device='cuda:0')


In [ ]:
from torch.optim import AdamW

EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights
)

print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Optimizer: AdamW")
print("Loss function: Weighted CrossEntropyLoss")

Epochs: 3
Learning rate: 2e-05
Optimizer: AdamW
Loss function: Weighted CrossEntropyLoss


In [ ]:
from tqdm.auto import tqdm

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        progress_bar.set_postfix(
            loss=loss.item(),
            accuracy=correct / total
        )

    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS}"
        f" - Loss: {epoch_loss:.4f}"
        f" - Accuracy: {epoch_accuracy:.4f}"
    )

Epoch 1/3:   0%|          | 0/1457 [00:00<?, ?it/s]


Epoch 1/3 - Loss: 0.1534 - Accuracy: 0.9618


Epoch 2/3:   0%|          | 0/1457 [00:00<?, ?it/s]


Epoch 2/3 - Loss: 0.0829 - Accuracy: 0.9772


Epoch 3/3:   0%|          | 0/1457 [00:00<?, ?it/s]


Epoch 3/3 - Loss: 0.0735 - Accuracy: 0.9839


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Testing"):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


# Accuracy
test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print("\n====================================")
print("mBERT DATASET A TEST RESULTS")
print("====================================")

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Classification report
print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Real (0)",
            "Fake (1)"
        ],
        digits=4
    )
)

# Confusion matrix
cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:")
print(cm)

Testing:   0%|          | 0/365 [00:00<?, ?it/s]


mBERT DATASET A TEST RESULTS
Test Accuracy: 0.9846
Test Accuracy: 98.46%

Classification Report:
              precision    recall  f1-score   support

    Real (0)     0.9897    0.9910    0.9904      2333
    Fake (1)     0.9636    0.9586    0.9611       580

    accuracy                         0.9846      2913
   macro avg     0.9767    0.9748    0.9757      2913
weighted avg     0.9845    0.9846    0.9845      2913


Confusion Matrix:
[[2312   21]
 [  24  556]]


In [ ]:
# ============================================================
# mBERT - DATASET B (IMBALANCED)
# ============================================================

import pandas as pd
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

from tqdm.auto import tqdm


# ============================================================
# 1. SETTINGS
# ============================================================

MODEL_NAME = "bert-base-multilingual-cased"

MAX_LENGTH = 200
BATCH_SIZE = 32

EPOCHS = 3
LEARNING_RATE = 2e-5

RANDOM_STATE = 42


# ============================================================
# 2. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 3. LOAD DATASET B
# ============================================================

df = pd.read_csv(
    "/content/Dataset_B_clean.csv"
)

print("\nDataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 4. SELECT COLUMNS
# ============================================================

TEXT_COLUMN = "News"
LABEL_COLUMN = "Authenticity"


# ============================================================
# 5. CLEAN DATA
# ============================================================

df = df.dropna(
    subset=[
        TEXT_COLUMN,
        LABEL_COLUMN
    ]
).copy()

df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)

df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(int)


# ============================================================
# 6. CLASS DISTRIBUTION
# ============================================================

print("\nClass distribution:")
print(
    df[LABEL_COLUMN].value_counts()
)

print("\nClass distribution (%):")
print(
    df[LABEL_COLUMN]
    .value_counts(normalize=True)
    .mul(100)
)


# ============================================================
# 7. TRAIN / TEST SPLIT
# ============================================================

X = df[TEXT_COLUMN].to_numpy()
y = df[LABEL_COLUMN].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\n====================================")
print("TRAIN / TEST SPLIT")
print("====================================")

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
print(
    pd.Series(y_train)
    .value_counts()
    .sort_index()
)

print("\nTesting class distribution:")
print(
    pd.Series(y_test)
    .value_counts()
    .sort_index()
)


# ============================================================
# 8. CLASS WEIGHTS
# ============================================================

classes = np.unique(y_train)

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(
    class_weights_array,
    dtype=torch.float
).to(device)

print("\nClass weights:")

for cls, weight in zip(
    classes,
    class_weights_array
):
    print(
        f"Class {cls}: {weight:.6f}"
    )


# ============================================================
# 9. LOAD mBERT TOKENIZER
# ============================================================

print("\nLoading mBERT tokenizer...")

tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# 10. PYTORCH DATASET
# ============================================================

class NewsDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length
    ):

        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(
                label,
                dtype=torch.long
            )
        }


# ============================================================
# 11. CREATE PYTORCH DATASETS
# ============================================================

train_dataset = NewsDataset(
    X_train,
    y_train,
    tokenizer,
    MAX_LENGTH
)

test_dataset = NewsDataset(
    X_test,
    y_test,
    tokenizer,
    MAX_LENGTH
)

print("\nPyTorch datasets created.")

print(
    "Training dataset:",
    len(train_dataset)
)

print(
    "Testing dataset:",
    len(test_dataset)
)


# ============================================================
# 12. CREATE DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\nTraining batches:", len(train_loader))
print("Testing batches:", len(test_loader))


# ============================================================
# 13. LOAD mBERT MODEL
# ============================================================

print("\nLoading mBERT model...")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print("\nModel loaded successfully.")


# ============================================================
# 14. OPTIMIZER
# ============================================================

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# 15. LOSS FUNCTION
# ============================================================

criterion = torch.nn.CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# 16. TRAINING
# ============================================================

print("\n====================================")
print("TRAINING mBERT ON DATASET B")
print("====================================")

training_history = []

model.train()

for epoch in range(EPOCHS):

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(
            outputs.logits,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        progress_bar.set_postfix(
            accuracy=f"{correct / total:.3f}",
            loss=f"{loss.item():.5f}"
        )

    epoch_loss = total_loss / len(train_loader)

    epoch_accuracy = correct / total

    training_history.append(
        {
            "epoch": epoch + 1,
            "loss": epoch_loss,
            "accuracy": epoch_accuracy
        }
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"- Loss: {epoch_loss:.4f} "
        f"- Accuracy: {epoch_accuracy:.4f}"
    )


# ============================================================
# 17. TESTING
# ============================================================

print("\n====================================")
print("TESTING mBERT ON DATASET B")
print("====================================")

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Testing"
    ):

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        labels = batch[
            "labels"
        ].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


# ============================================================
# 18. ACCURACY
# ============================================================

test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print("\n====================================")
print("mBERT DATASET B TEST RESULTS")
print("====================================")

print(
    f"Test Accuracy: {test_accuracy:.4f}"
)

print(
    f"Test Accuracy: {test_accuracy * 100:.2f}%"
)


# ============================================================
# 19. CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=[
            "Real (0)",
            "Fake (1)"
        ],
        digits=2
    )
)


# ============================================================
# 20. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:")
print(cm)


# ============================================================
# 21. TRAINING RESULTS
# ============================================================

print("\n====================================")
print("TRAINING RESULTS")
print("====================================")

for result in training_history:

    print(
        f"Epoch {result['epoch']}: "
        f"Loss = {result['loss']:.4f}, "
        f"Accuracy = {result['accuracy']:.4f}"
    )

Using device: cuda
GPU: Tesla T4

Dataset shape: (5226, 2)

Columns:
['News', 'Authenticity']

Class distribution:
Authenticity
1    2902
0    2324
Name: count, dtype: int64

Class distribution (%):
Authenticity
1    55.530042
0    44.469958
Name: proportion, dtype: float64

TRAIN / TEST SPLIT
Training samples: 4180
Testing samples: 1046

Training class distribution:
0    1859
1    2321
Name: count, dtype: int64

Testing class distribution:
0    465
1    581
Name: count, dtype: int64

Class weights:
Class 0: 1.124260
Class 1: 0.900474

Loading mBERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]


PyTorch datasets created.
Training dataset: 4180
Testing dataset: 1046

Training batches: 131
Testing batches: 33

Loading mBERT model...


...

Epoch 2/3 - Loss: 0.0665 - Accuracy: 0.9787


Epoch 3/3:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 3/3 - Loss: 0.0299 - Accuracy: 0.9911

TESTING mBERT ON DATASET B


Testing:   0%|          | 0/33 [00:00<?, ?it/s]


mBERT DATASET B TEST RESULTS
Test Accuracy: 0.9818
Test Accuracy: 98.18%

Classification Report:
              precision    recall  f1-score   support

    Real (0)       0.98      0.98      0.98       465
    Fake (1)       0.98      0.98      0.98       581

    accuracy                           0.98      1046
   macro avg       0.98      0.98      0.98      1046
weighted avg       0.98      0.98      0.98      1046


Confusion Matrix:
[[456   9]
 [ 10 571]]

TRAINING RESULTS
Epoch 1: Loss = 0.1848, Accuracy = 0.9352
Epoch 2: Loss = 0.0665, Accuracy = 0.9787
Epoch 3: Loss = 0.0299, Accuracy = 0.9911


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

MODEL_NAME = "bert-base-multilingual-cased"

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

DATASET_A_PATH = "/content/Dataset_A_clean.csv"
DATASET_B_PATH = "/content/Dataset_B_clean.csv"


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# LOAD DATASET A
# ============================================================

df_A = pd.read_csv(DATASET_A_PATH)

df_A = df_A[["Tamil", "Is Fake"]].copy()

df_A["Tamil"] = df_A["Tamil"].fillna("").astype(str)
df_A["Is Fake"] = df_A["Is Fake"].astype(int)


print("\n====================================")
print("DATASET A")
print("====================================")

print("Dataset A shape:", df_A.shape)

print("\nClass distribution:")
print(df_A["Is Fake"].value_counts())

print("\nClass distribution (%):")
print(
    df_A["Is Fake"]
    .value_counts(normalize=True)
    .mul(100)
)


# ============================================================
# LOAD DATASET B
# ============================================================

df_B = pd.read_csv(DATASET_B_PATH)

df_B = df_B[["News", "Authenticity"]].copy()

df_B["News"] = df_B["News"].fillna("").astype(str)
df_B["Authenticity"] = df_B["Authenticity"].astype(int)


print("\n====================================")
print("DATASET B")
print("====================================")

print("Dataset B shape:", df_B.shape)

print("\nClass distribution:")
print(df_B["Authenticity"].value_counts())

print("\nClass distribution (%):")
print(
    df_B["Authenticity"]
    .value_counts(normalize=True)
    .mul(100)
)


# ============================================================
# GENERALISATION SETUP
#
# IMPORTANT:
#
# NO 80:20 SPLIT
#
# 100% Dataset A = TRAINING
# 100% Dataset B = EXTERNAL TESTING
# ============================================================

X_train = df_A["Tamil"].tolist()
y_train = df_A["Is Fake"].tolist()

X_test = df_B["News"].tolist()
y_test = df_B["Authenticity"].tolist()


print("\n====================================")
print("GENERALISATION SETUP: A → B")
print("====================================")

print(
    "Training samples (Dataset A):",
    len(X_train)
)

print(
    "External testing samples (Dataset B):",
    len(X_test)
)


# ============================================================
# CLASS WEIGHTS
#
# Calculate ONLY using Dataset A
# ============================================================

classes = np.array([0, 1])

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(y_train)
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float
).to(device)


print("\nClass weights:")
print(
    f"Class 0: {class_weights[0].item():.6f}"
)

print(
    f"Class 1: {class_weights[1].item():.6f}"
)


# ============================================================
# LOAD mBERT TOKENIZER
# ============================================================

print("\nLoading mBERT tokenizer...")

tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# CREATE HUGGING FACE DATASETS
# ============================================================

train_df = pd.DataFrame({
    "text": X_train,
    "labels": y_train
})

test_df = pd.DataFrame({
    "text": X_test,
    "labels": y_test
})


train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)


# ============================================================
# TOKENIZATION
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )


print("\nTokenizing Dataset A...")

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    desc="Dataset A tokenization"
)


print("\nTokenizing Dataset B...")

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    desc="Dataset B tokenization"
)


# ============================================================
# REMOVE TEXT COLUMN
# ============================================================

train_dataset = train_dataset.remove_columns(
    ["text"]
)

test_dataset = test_dataset.remove_columns(
    ["text"]
)


# ============================================================
# SET TORCH FORMAT
# ============================================================

train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)


print("\nPyTorch datasets created.")

print(
    "Training dataset:",
    len(train_dataset)
)

print(
    "Testing dataset:",
    len(test_dataset)
)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("\nTraining batches:", len(train_loader))
print("Testing batches:", len(test_loader))


# ============================================================
# LOAD mBERT MODEL
# ============================================================

print("\nLoading mBERT model...")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print("Model loaded successfully.")


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# TRAINING
# ============================================================

print("\n====================================")
print("TRAINING mBERT ON DATASET A")
print("GENERALISATION EXPERIMENT: A → B")
print("====================================")


training_results = []


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0


    progress_bar = tqdm(
        train_loader,
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        unit="batch"
    )


    for batch in progress_bar:

        # ----------------------------------------------------
        # MOVE DATA TO GPU
        # ----------------------------------------------------

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )


        # ----------------------------------------------------
        # CLEAR GRADIENTS
        # ----------------------------------------------------

        optimizer.zero_grad()


        # ----------------------------------------------------
        # FORWARD PASS
        # ----------------------------------------------------

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits


        # ----------------------------------------------------
        # LOSS
        # ----------------------------------------------------

        loss = criterion(
            logits,
            labels
        )


        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # UPDATE MODEL
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # CALCULATE TRAINING ACCURACY
        # ----------------------------------------------------

        predictions = torch.argmax(
            logits,
            dim=1
        )

        batch_correct = (
            predictions == labels
        ).sum().item()

        correct += batch_correct

        total += labels.size(0)

        total_loss += loss.item()


        # ----------------------------------------------------
        # UPDATE PROGRESS BAR
        # ----------------------------------------------------

        current_accuracy = correct / total

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{current_accuracy:.4f}"
        )


    # ========================================================
    # EPOCH RESULTS
    # ========================================================

    epoch_loss = (
        total_loss / len(train_loader)
    )

    epoch_accuracy = (
        correct / total
    )


    training_results.append({
        "epoch": epoch + 1,
        "loss": epoch_loss,
        "accuracy": epoch_accuracy
    })


    print(
        f"\nEpoch {epoch + 1}/{EPOCHS} "
        f"- Loss: {epoch_loss:.4f} "
        f"- Accuracy: {epoch_accuracy:.4f}"
    )


# ============================================================
# EXTERNAL TESTING ON DATASET B
# ============================================================

print("\n====================================")
print("GENERALISATION TEST")
print("mBERT TRAINED ON DATASET A")
print("TESTED ON 100% DATASET B")
print("====================================")


model.eval()

all_predictions = []
all_labels = []


with torch.no_grad():

    progress_bar = tqdm(
        test_loader,
        total=len(test_loader),
        desc="Testing on Dataset B",
        unit="batch"
    )


    for batch in progress_bar:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        logits = outputs.logits


        predictions = torch.argmax(
            logits,
            dim=1
        )


        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )


# ============================================================
# TEST ACCURACY
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

report = classification_report(
    all_labels,
    all_predictions,
    target_names=[
        "Real (0)",
        "Fake (1)"
    ],
    digits=4
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n====================================")
print("mBERT DATASET A → DATASET B RESULTS")
print("====================================")

print(
    f"External Test Accuracy: {accuracy:.4f}"
)

print(
    f"External Test Accuracy: {accuracy * 100:.2f}%"
)


print("\nClassification Report:")
print(report)


print("Confusion Matrix:")
print(cm)


# ============================================================
# TRAINING RESULTS
# ============================================================

print("\n====================================")
print("TRAINING RESULTS")
print("====================================")


for result in training_results:

    print(
        f"Epoch {result['epoch']}: "
        f"Loss = {result['loss']:.4f}, "
        f"Accuracy = {result['accuracy']:.4f}"
    )


# ============================================================
# GPU MEMORY
# ============================================================

if torch.cuda.is_available():

    print("\n====================================")
    print("GPU MEMORY")
    print("====================================")

    allocated = (
        torch.cuda.memory_allocated()
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved()
        / 1024**3
    )

    print(
        f"Allocated GPU memory: "
        f"{allocated:.2f} GB"
    )

    print(
        f"Reserved GPU memory: "
        f"{reserved:.2f} GB"
    )

Using device: cuda
GPU: Tesla T4

DATASET A
Dataset A shape: (14564, 2)

Class distribution:
Is Fake
0    11662
1     2902
Name: count, dtype: int64

Class distribution (%):
Is Fake
0    80.074155
1    19.925845
Name: proportion, dtype: float64

DATASET B
Dataset B shape: (5226, 2)

Class distribution:
Authenticity
1    2902
0    2324
Name: count, dtype: int64

Class distribution (%):
Authenticity
1    55.530042
0    44.469958
Name: proportion, dtype: float64

GENERALISATION SETUP: A → B
Training samples (Dataset A): 14564
External testing samples (Dataset B): 5226

Class weights:
Class 0: 0.624421
Class 1: 2.509304

Loading mBERT tokenizer...

Tokenizing Dataset A...


Dataset A tokenization:   0%|          | 0/14564 [00:00<?, ? examples/s]


Tokenizing Dataset B...


Dataset B tokenization:   0%|          | 0/5226 [00:00<?, ? examples/s]


PyTorch datasets created.
Training dataset: 14564
Testing dataset: 5226

Training batches: 911
Testing batches: 327

Loading mBERT model...


...


Epoch 2/3 - Loss: 0.0688 - Accuracy: 0.9836


Epoch 3/3:   0%|          | 0/911 [00:00<?, ?batch/s]


Epoch 3/3 - Loss: 0.0416 - Accuracy: 0.9902

GENERALISATION TEST
mBERT TRAINED ON DATASET A
TESTED ON 100% DATASET B


Testing on Dataset B:   0%|          | 0/327 [00:00<?, ?batch/s]


mBERT DATASET A → DATASET B RESULTS
External Test Accuracy: 0.9923
External Test Accuracy: 99.23%

Classification Report:
              precision    recall  f1-score   support

    Real (0)     0.9939    0.9888    0.9914      2324
    Fake (1)     0.9911    0.9952    0.9931      2902

    accuracy                         0.9923      5226
   macro avg     0.9925    0.9920    0.9922      5226
weighted avg     0.9924    0.9923    0.9923      5226

Confusion Matrix:
[[2298   26]
 [  14 2888]]

TRAINING RESULTS
Epoch 1: Loss = 0.1358, Accuracy = 0.9651
Epoch 2: Loss = 0.0688, Accuracy = 0.9836
Epoch 3: Loss = 0.0416, Accuracy = 0.9902

GPU MEMORY
Allocated GPU memory: 2.68 GB
Reserved GPU memory: 4.20 GB


In [ ]:
# ============================================================
# mBERT GENERALISATION EXPERIMENT: DATASET B → DATASET A
#
# 100% Dataset B = TRAINING
# 100% Dataset A = EXTERNAL TESTING
#
# NO 80:20 TRAIN/TEST SPLIT
# ============================================================


import os
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

MODEL_NAME = "bert-base-multilingual-cased"

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5


# ============================================================
# DATASET PATHS
# ============================================================

DATASET_A_PATH = "/content/Dataset_A_clean.csv"
DATASET_B_PATH = "/content/Dataset_B_clean.csv"


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# LOAD DATASET A
# ============================================================

df_A = pd.read_csv(
    DATASET_A_PATH
)

df_A = df_A[
    ["Tamil", "Is Fake"]
].copy()

df_A["Tamil"] = (
    df_A["Tamil"]
    .fillna("")
    .astype(str)
)

df_A["Is Fake"] = (
    df_A["Is Fake"]
    .astype(int)
)


print("\n====================================")
print("DATASET A")
print("====================================")

print(
    "Dataset A shape:",
    df_A.shape
)

print("\nClass distribution:")

print(
    df_A["Is Fake"]
    .value_counts()
)

print("\nClass distribution (%):")

print(
    df_A["Is Fake"]
    .value_counts(
        normalize=True
    )
    .mul(100)
)


# ============================================================
# LOAD DATASET B
# ============================================================

df_B = pd.read_csv(
    DATASET_B_PATH
)

df_B = df_B[
    ["News", "Authenticity"]
].copy()

df_B["News"] = (
    df_B["News"]
    .fillna("")
    .astype(str)
)

df_B["Authenticity"] = (
    df_B["Authenticity"]
    .astype(int)
)


print("\n====================================")
print("DATASET B")
print("====================================")

print(
    "Dataset B shape:",
    df_B.shape
)

print("\nClass distribution:")

print(
    df_B["Authenticity"]
    .value_counts()
)

print("\nClass distribution (%):")

print(
    df_B["Authenticity"]
    .value_counts(
        normalize=True
    )
    .mul(100)
)


# ============================================================
# GENERALISATION SETUP: B → A
#
# IMPORTANT:
#
# Dataset B = 100% TRAINING
# Dataset A = 100% EXTERNAL TESTING
#
# NO TRAIN/TEST SPLIT
# ============================================================

X_train = df_B["News"].tolist()

y_train = df_B[
    "Authenticity"
].tolist()


X_test = df_A["Tamil"].tolist()

y_test = df_A[
    "Is Fake"
].tolist()


print("\n====================================")
print("GENERALISATION SETUP: B → A")
print("====================================")

print(
    "Training samples (Dataset B):",
    len(X_train)
)

print(
    "External testing samples (Dataset A):",
    len(X_test)
)


# ============================================================
# CLASS WEIGHTS
#
# IMPORTANT:
# Calculate ONLY from Dataset B
# because Dataset B is the training dataset.
# ============================================================

classes = np.array(
    [0, 1]
)

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(y_train)
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float
).to(device)


print("\nClass weights:")

print(
    f"Class 0: "
    f"{class_weights[0].item():.6f}"
)

print(
    f"Class 1: "
    f"{class_weights[1].item():.6f}"
)


# ============================================================
# LOAD mBERT TOKENIZER
# ============================================================

print("\nLoading mBERT tokenizer...")

tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# CREATE HUGGING FACE DATASETS
# ============================================================

train_df = pd.DataFrame({
    "text": X_train,
    "labels": y_train
})


test_df = pd.DataFrame({
    "text": X_test,
    "labels": y_test
})


train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)


test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)


# ============================================================
# TOKENIZATION FUNCTION
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )


# ============================================================
# TOKENIZE DATASET B
# ============================================================

print("\nTokenizing Dataset B...")

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    desc="Dataset B tokenization"
)


# ============================================================
# TOKENIZE DATASET A
# ============================================================

print("\nTokenizing Dataset A...")

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    desc="Dataset A tokenization"
)


# ============================================================
# REMOVE ORIGINAL TEXT COLUMN
# ============================================================

train_dataset = train_dataset.remove_columns(
    ["text"]
)

test_dataset = test_dataset.remove_columns(
    ["text"]
)


# ============================================================
# SET PYTORCH FORMAT
# ============================================================

train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)


test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)


print("\nPyTorch datasets created.")

print(
    "Training dataset:",
    len(train_dataset)
)

print(
    "Testing dataset:",
    len(test_dataset)
)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("\nTraining batches:")

print(
    len(train_loader)
)


print("Testing batches:")

print(
    len(test_loader)
)


# ============================================================
# LOAD mBERT MODEL
# ============================================================

print("\nLoading mBERT model...")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

print("Model loaded successfully.")


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# TRAINING
# ============================================================

print("\n====================================")
print("TRAINING mBERT ON DATASET B")
print("GENERALISATION EXPERIMENT: B → A")
print("====================================")


training_results = []


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    correct = 0

    total = 0


    progress_bar = tqdm(
        train_loader,
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        unit="batch"
    )


    for batch in progress_bar:

        # ----------------------------------------------------
        # MOVE BATCH TO GPU
        # ----------------------------------------------------

        input_ids = batch[
            "input_ids"
        ].to(
            device,
            non_blocking=True
        )


        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )


        labels = batch[
            "labels"
        ].to(
            device,
            non_blocking=True
        )


        # ----------------------------------------------------
        # CLEAR GRADIENTS
        # ----------------------------------------------------

        optimizer.zero_grad()


        # ----------------------------------------------------
        # FORWARD PASS
        # ----------------------------------------------------

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        logits = outputs.logits


        # ----------------------------------------------------
        # CALCULATE LOSS
        # ----------------------------------------------------

        loss = criterion(
            logits,
            labels
        )


        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # UPDATE MODEL
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # TRAINING ACCURACY
        # ----------------------------------------------------

        predictions = torch.argmax(
            logits,
            dim=1
        )


        batch_correct = (
            predictions == labels
        ).sum().item()


        correct += batch_correct

        total += labels.size(0)

        total_loss += loss.item()


        current_accuracy = (
            correct / total
        )


        # ----------------------------------------------------
        # UPDATE PROGRESS BAR
        # ----------------------------------------------------

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{current_accuracy:.4f}"
        )


    # ========================================================
    # EPOCH RESULTS
    # ========================================================

    epoch_loss = (
        total_loss /
        len(train_loader)
    )


    epoch_accuracy = (
        correct /
        total
    )


    training_results.append({
        "epoch": epoch + 1,
        "loss": epoch_loss,
        "accuracy": epoch_accuracy
    })


    print(
        f"\nEpoch {epoch + 1}/{EPOCHS} "
        f"- Loss: {epoch_loss:.4f} "
        f"- Accuracy: {epoch_accuracy:.4f}"
    )


# ============================================================
# EXTERNAL TESTING
#
# Dataset A has NEVER been used for training.
# ============================================================

print("\n====================================")
print("GENERALISATION TEST")
print("mBERT TRAINED ON DATASET B")
print("TESTED ON 100% DATASET A")
print("====================================")


model.eval()


all_predictions = []

all_labels = []


with torch.no_grad():

    progress_bar = tqdm(
        test_loader,
        total=len(test_loader),
        desc="Testing on Dataset A",
        unit="batch"
    )


    for batch in progress_bar:

        # ----------------------------------------------------
        # MOVE BATCH TO GPU
        # ----------------------------------------------------

        input_ids = batch[
            "input_ids"
        ].to(
            device,
            non_blocking=True
        )


        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )


        labels = batch[
            "labels"
        ].to(
            device,
            non_blocking=True
        )


        # ----------------------------------------------------
        # MODEL PREDICTION
        # ----------------------------------------------------

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        logits = outputs.logits


        predictions = torch.argmax(
            logits,
            dim=1
        )


        # ----------------------------------------------------
        # SAVE PREDICTIONS
        # ----------------------------------------------------

        all_predictions.extend(
            predictions
            .cpu()
            .numpy()
        )


        all_labels.extend(
            labels
            .cpu()
            .numpy()
        )


# ============================================================
# TEST ACCURACY
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

report = classification_report(
    all_labels,
    all_predictions,
    target_names=[
        "Real (0)",
        "Fake (1)"
    ],
    digits=2
)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n====================================")
print("mBERT DATASET B → DATASET A RESULTS")
print("====================================")


print(
    f"External Test Accuracy: "
    f"{accuracy:.4f}"
)


print(
    f"External Test Accuracy: "
    f"{accuracy * 100:.2f}%"
)


print("\nClassification Report:")

print(report)


print("Confusion Matrix:")

print(cm)


# ============================================================
# TRAINING RESULTS
# ============================================================

print("\n====================================")
print("TRAINING RESULTS")
print("====================================")


for result in training_results:

    print(
        f"Epoch {result['epoch']}: "
        f"Loss = {result['loss']:.4f}, "
        f"Accuracy = {result['accuracy']:.4f}"
    )


# ============================================================
# GPU MEMORY
# ============================================================

if torch.cuda.is_available():

    print("\n====================================")
    print("GPU MEMORY")
    print("====================================")


    allocated = (
        torch.cuda.memory_allocated()
        / 1024**3
    )


    reserved = (
        torch.cuda.memory_reserved()
        / 1024**3
    )


    print(
        f"Allocated GPU memory: "
        f"{allocated:.2f} GB"
    )


    print(
        f"Reserved GPU memory: "
        f"{reserved:.2f} GB"
    )

Using device: cuda
GPU: Tesla T4

DATASET A
Dataset A shape: (14564, 2)

Class distribution:
Is Fake
0    11662
1     2902
Name: count, dtype: int64

Class distribution (%):
Is Fake
0    80.074155
1    19.925845
Name: proportion, dtype: float64

DATASET B
Dataset B shape: (5226, 2)

Class distribution:
Authenticity
1    2902
0    2324
Name: count, dtype: int64

Class distribution (%):
Authenticity
1    55.530042
0    44.469958
Name: proportion, dtype: float64

GENERALISATION SETUP: B → A
Training samples (Dataset B): 5226
External testing samples (Dataset A): 14564

Class weights:
Class 0: 1.124355
Class 1: 0.900414

Loading mBERT tokenizer...

Tokenizing Dataset B...


Dataset B tokenization:   0%|          | 0/5226 [00:00<?, ? examples/s]


Tokenizing Dataset A...


Dataset A tokenization:   0%|          | 0/14564 [00:00<?, ? examples/s]


PyTorch datasets created.
Training dataset: 5226
Testing dataset: 14564

Training batches:
327
Testing batches:
911

Loading mBERT model...


...


Epoch 2/3 - Loss: 0.0722 - Accuracy: 0.9761


Epoch 3/3:   0%|          | 0/327 [00:00<?, ?batch/s]


Epoch 3/3 - Loss: 0.0409 - Accuracy: 0.9868

GENERALISATION TEST
mBERT TRAINED ON DATASET B
TESTED ON 100% DATASET A


Testing on Dataset A:   0%|          | 0/911 [00:00<?, ?batch/s]


mBERT DATASET B → DATASET A RESULTS
External Test Accuracy: 0.8943
External Test Accuracy: 89.43%

Classification Report:
              precision    recall  f1-score   support

    Real (0)       1.00      0.87      0.93     11662
    Fake (1)       0.65      0.99      0.79      2902

    accuracy                           0.89     14564
   macro avg       0.83      0.93      0.86     14564
weighted avg       0.93      0.89      0.90     14564

Confusion Matrix:
[[10144  1518]
 [   21  2881]]

TRAINING RESULTS
Epoch 1: Loss = 0.1481, Accuracy = 0.9468
Epoch 2: Loss = 0.0722, Accuracy = 0.9761
Epoch 3: Loss = 0.0409, Accuracy = 0.9868

GPU MEMORY
Allocated GPU memory: 2.68 GB
Reserved GPU memory: 4.22 GB


In [ ]:
# ============================================================
# mBERT EXPERIMENT 5
# COMBINED DATASET - IMBALANCED
#
# Dataset A + Dataset B
# 80:20 STRATIFIED TRAIN / TEST SPLIT
# NO BALANCING
# NO DUPLICATE REMOVAL AFTER COMBINING
# ============================================================

import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

MODEL_NAME = "bert-base-multilingual-cased"

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

DATASET_A_PATH = "/content/Dataset_A_clean.csv"
DATASET_B_PATH = "/content/Dataset_B_clean.csv"


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# DATASET A
# ============================================================

df_A = pd.read_csv(
    DATASET_A_PATH
)

df_A = df_A[
    ["Tamil", "Is Fake"]
].copy()

df_A["Tamil"] = (
    df_A["Tamil"]
    .fillna("")
    .astype(str)
)

df_A["Is Fake"] = (
    df_A["Is Fake"]
    .astype(int)
)


print("\n====================================")
print("DATASET A")
print("====================================")

print(
    "Dataset A shape:",
    df_A.shape
)

print("\nClass distribution:")

print(
    df_A["Is Fake"]
    .value_counts()
    .sort_index()
)

print("\nClass distribution (%):")

print(
    df_A["Is Fake"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


# ============================================================
# DATASET B
# ============================================================

df_B = pd.read_csv(
    DATASET_B_PATH
)

df_B = df_B[
    ["News", "Authenticity"]
].copy()

df_B["News"] = (
    df_B["News"]
    .fillna("")
    .astype(str)
)

df_B["Authenticity"] = (
    df_B["Authenticity"]
    .astype(int)
)


print("\n====================================")
print("DATASET B")
print("====================================")

print(
    "Dataset B shape:",
    df_B.shape
)

print("\nClass distribution:")

print(
    df_B["Authenticity"]
    .value_counts()
    .sort_index()
)

print("\nClass distribution (%):")

print(
    df_B["Authenticity"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


# ============================================================
# STANDARDISE DATASET A
# ============================================================

combined_A = pd.DataFrame({
    "text": df_A["Tamil"],
    "labels": df_A["Is Fake"]
})


# ============================================================
# STANDARDISE DATASET B
# ============================================================

combined_B = pd.DataFrame({
    "text": df_B["News"],
    "labels": df_B["Authenticity"]
})


# ============================================================
# COMBINE DATASETS
#
# IMPORTANT:
# DO NOT REMOVE DUPLICATES HERE.
#
# Dataset A = 14,564
# Dataset B = 5,226
#
# Total = 19,790
# ============================================================

combined_df = pd.concat(
    [
        combined_A,
        combined_B
    ],
    ignore_index=True
)


# ============================================================
# COMBINED DATASET INFORMATION
# ============================================================

print("\n====================================")
print("COMBINED DATASET - IMBALANCED")
print("====================================")

print(
    "Combined dataset shape:",
    combined_df.shape
)

print(
    "\nExpected total:",
    14564 + 5226
)

print("\nClass distribution:")

combined_counts = (
    combined_df["labels"]
    .value_counts()
    .sort_index()
)

print(combined_counts)


print("\nClass distribution (%):")

combined_percent = (
    combined_df["labels"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)

print(combined_percent)


# ============================================================
# CHECK TOTAL
# ============================================================

assert len(combined_df) == 19790, (
    f"Expected 19790 records, "
    f"but found {len(combined_df)}"
)

assert combined_counts[0] == 13986, (
    f"Expected 13986 real samples, "
    f"but found {combined_counts[0]}"
)

assert combined_counts[1] == 5804, (
    f"Expected 5804 fake samples, "
    f"but found {combined_counts[1]}"
)


print("\nDataset size verification: PASSED")


# ============================================================
# 80:20 STRATIFIED TRAIN / TEST SPLIT
#
# This is NOT a generalisation experiment.
#
# Combined dataset:
# 19,790 total
#
# Training:
# 15,832
#
# Testing:
# 3,958
# ============================================================

X = combined_df["text"].values

y = combined_df["labels"].values


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)


# ============================================================
# TRAIN / TEST INFORMATION
# ============================================================

print("\n====================================")
print("TRAIN / TEST SPLIT")
print("====================================")

print(
    "Training samples:",
    len(X_train)
)

print(
    "Testing samples:",
    len(X_test)
)


print("\nTraining class distribution:")

train_counts = (
    pd.Series(y_train)
    .value_counts()
    .sort_index()
)

print(train_counts)


print("\nTesting class distribution:")

test_counts = (
    pd.Series(y_test)
    .value_counts()
    .sort_index()
)

print(test_counts)


print("\nTraining class distribution (%):")

print(
    pd.Series(y_train)
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


print("\nTesting class distribution (%):")

print(
    pd.Series(y_test)
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


# ============================================================
# VERIFY EXPECTED TEST SUPPORT
# ============================================================

assert len(X_test) == 3958, (
    f"Expected 3958 test samples, "
    f"but found {len(X_test)}"
)

assert test_counts[0] == 2797, (
    f"Expected 2797 Real samples, "
    f"but found {test_counts[0]}"
)

assert test_counts[1] == 1161, (
    f"Expected 1161 Fake samples, "
    f"but found {test_counts[1]}"
)


print("\nTest-set verification: PASSED")


# ============================================================
# CLASS WEIGHTS
#
# Calculate ONLY from training data.
# ============================================================

classes = np.array([0, 1])

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(y_train)
)


class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float
).to(device)


print("\nClass weights:")

print(
    f"Class 0: "
    f"{class_weights[0].item():.6f}"
)

print(
    f"Class 1: "
    f"{class_weights[1].item():.6f}"
)


# ============================================================
# LOAD mBERT TOKENIZER
# ============================================================

print("\nLoading mBERT tokenizer...")

tokenizer = BertTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================
# CREATE HUGGING FACE DATASETS
# ============================================================

train_df = pd.DataFrame({
    "text": X_train,
    "labels": y_train
})


test_df = pd.DataFrame({
    "text": X_test,
    "labels": y_test
})


train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)


test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)


# ============================================================
# TOKENIZATION
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )


# ============================================================
# TOKENIZE TRAINING DATA
# ============================================================

print("\nTokenizing training data...")

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    desc="Training tokenization"
)


# ============================================================
# TOKENIZE TEST DATA
# ============================================================

print("\nTokenizing testing data...")

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
    desc="Testing tokenization"
)


# ============================================================
# REMOVE ORIGINAL TEXT COLUMN
# ============================================================

train_dataset = train_dataset.remove_columns(
    ["text"]
)

test_dataset = test_dataset.remove_columns(
    ["text"]
)


# ============================================================
# PYTORCH FORMAT
# ============================================================

train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)


test_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)


print("\nPyTorch datasets created.")

print(
    "Training dataset:",
    len(train_dataset)
)

print(
    "Testing dataset:",
    len(test_dataset)
)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print("\nTraining batches:")

print(
    len(train_loader)
)

print("Testing batches:")

print(
    len(test_loader)
)


# ============================================================
# LOAD mBERT MODEL
# ============================================================

print("\nLoading mBERT model...")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


model.to(device)


print("Model loaded successfully.")


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = CrossEntropyLoss(
    weight=class_weights
)


# ============================================================
# TRAINING
# ============================================================

print("\n====================================")
print("TRAINING mBERT")
print("COMBINED DATASET - IMBALANCED")
print("====================================")


training_results = []


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    correct = 0

    total = 0


    progress_bar = tqdm(
        train_loader,
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        unit="batch"
    )


    for batch in progress_bar:

        input_ids = batch[
            "input_ids"
        ].to(
            device,
            non_blocking=True
        )


        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )


        labels = batch[
            "labels"
        ].to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad()


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        logits = outputs.logits


        loss = criterion(
            logits,
            labels
        )


        loss.backward()

        optimizer.step()


        predictions = torch.argmax(
            logits,
            dim=1
        )


        batch_correct = (
            predictions == labels
        ).sum().item()


        correct += batch_correct

        total += labels.size(0)

        total_loss += loss.item()


        current_accuracy = (
            correct / total
        )


        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{current_accuracy:.4f}"
        )


    epoch_loss = (
        total_loss /
        len(train_loader)
    )


    epoch_accuracy = (
        correct /
        total
    )


    training_results.append({
        "epoch": epoch + 1,
        "loss": epoch_loss,
        "accuracy": epoch_accuracy
    })


    print(
        f"\nEpoch {epoch + 1}/{EPOCHS} "
        f"- Loss: {epoch_loss:.4f} "
        f"- Accuracy: {epoch_accuracy:.4f}"
    )


# ============================================================
# TESTING
# ============================================================

print("\n====================================")
print("TESTING mBERT")
print("COMBINED DATASET - IMBALANCED")
print("====================================")


model.eval()


all_predictions = []

all_labels = []


with torch.no_grad():

    progress_bar = tqdm(
        test_loader,
        total=len(test_loader),
        desc="Testing",
        unit="batch"
    )


    for batch in progress_bar:

        input_ids = batch[
            "input_ids"
        ].to(
            device,
            non_blocking=True
        )


        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )


        labels = batch[
            "labels"
        ].to(
            device,
            non_blocking=True
        )


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        logits = outputs.logits


        predictions = torch.argmax(
            logits,
            dim=1
        )


        all_predictions.extend(
            predictions
            .cpu()
            .numpy()
        )


        all_labels.extend(
            labels
            .cpu()
            .numpy()
        )


# ============================================================
# FINAL METRICS
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)


report = classification_report(
    all_labels,
    all_predictions,
    target_names=[
        "Real (0)",
        "Fake (1)"
    ],
    digits=4
)


cm = confusion_matrix(
    all_labels,
    all_predictions
)


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n====================================")
print("mBERT COMBINED DATASET")
print("IMBALANCED TEST RESULTS")
print("====================================")


print(
    f"Test Accuracy: {accuracy:.4f}"
)


print(
    f"Test Accuracy: {accuracy * 100:.2f}%"
)


print("\nClassification Report:")

print(report)


print("Confusion Matrix:")

print(cm)


# ============================================================
# TRAINING RESULTS
# ============================================================

print("\n====================================")
print("TRAINING RESULTS")
print("====================================")


for result in training_results:

    print(
        f"Epoch {result['epoch']}: "
        f"Loss = {result['loss']:.4f}, "
        f"Accuracy = {result['accuracy']:.4f}"
    )


# ============================================================
# GPU MEMORY
# ============================================================

if torch.cuda.is_available():

    print("\n====================================")
    print("GPU MEMORY")
    print("====================================")


    allocated = (
        torch.cuda.memory_allocated()
        / 1024**3
    )


    reserved = (
        torch.cuda.memory_reserved()
        / 1024**3
    )


    print(
        f"Allocated GPU memory: "
        f"{allocated:.2f} GB"
    )


    print(
        f"Reserved GPU memory: "
        f"{reserved:.2f} GB"
    )


# ============================================================
# END
# ============================================================

print("\n====================================")
print("EXPERIMENT 5 COMPLETED")
print("mBERT - COMBINED DATASET - IMBALANCED")
print("====================================")

Using device: cuda
GPU: Tesla T4

DATASET A
Dataset A shape: (14564, 2)

Class distribution:
Is Fake
0    11662
1     2902
Name: count, dtype: int64

Class distribution (%):
Is Fake
0    80.074155
1    19.925845
Name: proportion, dtype: float64

DATASET B
Dataset B shape: (5226, 2)

Class distribution:
Authenticity
0    2324
1    2902
Name: count, dtype: int64

Class distribution (%):
Authenticity
0    44.469958
1    55.530042
Name: proportion, dtype: float64

COMBINED DATASET - IMBALANCED
Combined dataset shape: (19790, 2)

Expected total: 19790

Class distribution:
labels
0    13986
1     5804
Name: count, dtype: int64

Class distribution (%):
labels
0    70.672057
1    29.327943
Name: proportion, dtype: float64

Dataset size verification: PASSED

TRAIN / TEST SPLIT
Training samples: 15832
Testing samples: 3958

Training class distribution:
0    11189
1     4643
Name: count, dtype: int64

Testing class distribution:
0    2797
1    1161
Name: count, dtype: int64

Training class distri

Training tokenization:   0%|          | 0/15832 [00:00<?, ? examples/s]


Tokenizing testing data...


Testing tokenization:   0%|          | 0/3958 [00:00<?, ? examples/s]


PyTorch datasets created.
Training dataset: 15832
Testing dataset: 3958

Training batches:
990
Testing batches:
248

Loading mBERT model...


...


Epoch 2/3 - Loss: 0.0503 - Accuracy: 0.9883


Epoch 3/3:   0%|          | 0/990 [00:00<?, ?batch/s]


Epoch 3/3 - Loss: 0.0299 - Accuracy: 0.9926

TESTING mBERT
COMBINED DATASET - IMBALANCED


Testing:   0%|          | 0/248 [00:00<?, ?batch/s]


mBERT COMBINED DATASET
IMBALANCED TEST RESULTS
Test Accuracy: 0.9937
Test Accuracy: 99.37%

Classification Report:
              precision    recall  f1-score   support

    Real (0)     0.9968    0.9943    0.9955      2797
    Fake (1)     0.9863    0.9922    0.9893      1161

    accuracy                         0.9937      3958
   macro avg     0.9915    0.9933    0.9924      3958
weighted avg     0.9937    0.9937    0.9937      3958

Confusion Matrix:
[[2781   16]
 [   9 1152]]

TRAINING RESULTS
Epoch 1: Loss = 0.1178, Accuracy = 0.9667
Epoch 2: Loss = 0.0503, Accuracy = 0.9883
Epoch 3: Loss = 0.0299, Accuracy = 0.9926

GPU MEMORY
Allocated GPU memory: 2.69 GB
Reserved GPU memory: 4.22 GB

EXPERIMENT 5 COMPLETED
mBERT - COMBINED DATASET - IMBALANCED


In [15]:
# ============================================================
# mBERT EXPERIMENT 6
# COMBINED DATASET - BALANCED
#
# Dataset A + Dataset B
#
# IMPORTANT:
# - DO NOT REMOVE DUPLICATES
# - BALANCE ENTIRE COMBINED DATASET FIRST
# - THEN 80:20 STRATIFIED TRAIN / TEST SPLIT
#
# EXPECTED:
#
# Original Combined:
# Class 0 = 13,986
# Class 1 = 5,804
#
# Balanced Combined:
# Class 0 = 5,804
# Class 1 = 5,804
# Total    = 11,608
#
# Training:
# Class 0 = 4,643
# Class 1 = 4,643
# Total    = 9,286
#
# Testing:
# Class 0 = 1,161
# Class 1 = 1,161
# Total    = 2,322
#
# MODEL:
# bert-base-multilingual-cased (mBERT)
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

from datasets import Dataset

from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tqdm.auto import tqdm


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

MODEL_NAME = "bert-base-multilingual-cased"

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

DATASET_A_PATH = "/content/Dataset_A_clean.csv"
DATASET_B_PATH = "/content/Dataset_B_clean.csv"


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# LOAD DATASET A
# ============================================================

df_A = pd.read_csv(
    DATASET_A_PATH
)

df_A = df_A[
    ["Tamil", "Is Fake"]
].copy()

df_A["Tamil"] = (
    df_A["Tamil"]
    .fillna("")
    .astype(str)
)

df_A["Is Fake"] = (
    df_A["Is Fake"]
    .astype(int)
)


print("\n====================================")
print("DATASET A")
print("====================================")

print(
    "Dataset A shape:",
    df_A.shape
)

print("\nClass distribution:")

print(
    df_A["Is Fake"]
    .value_counts()
    .sort_index()
)

print("\nClass distribution (%):")

print(
    df_A["Is Fake"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


# ============================================================
# LOAD DATASET B
# ============================================================

df_B = pd.read_csv(
    DATASET_B_PATH
)

df_B = df_B[
    ["News", "Authenticity"]
].copy()

df_B["News"] = (
    df_B["News"]
    .fillna("")
    .astype(str)
)

df_B["Authenticity"] = (
    df_B["Authenticity"]
    .astype(int)
)


print("\n====================================")
print("DATASET B")
print("====================================")

print(
    "Dataset B shape:",
    df_B.shape
)

print("\nClass distribution:")

print(
    df_B["Authenticity"]
    .value_counts()
    .sort_index()
)

print("\nClass distribution (%):")

print(
    df_B["Authenticity"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


# ============================================================
# STANDARDISE COLUMN NAMES
# ============================================================

combined_A = pd.DataFrame({

    "text": df_A["Tamil"],

    "labels": df_A["Is Fake"]

})


combined_B = pd.DataFrame({

    "text": df_B["News"],

    "labels": df_B["Authenticity"]

})


# ============================================================
# COMBINE DATASETS
#
# IMPORTANT:
# NO DROP_DUPLICATES()
# ============================================================

combined_df = pd.concat(

    [
        combined_A,
        combined_B
    ],

    ignore_index=True

)


# ============================================================
# COMBINED DATASET INFORMATION
# ============================================================

print("\n====================================")
print("COMBINED DATASET - IMBALANCED")
print("====================================")

print(
    "Combined dataset shape:",
    combined_df.shape
)

print("\nClass distribution:")

combined_counts = (
    combined_df["labels"]
    .value_counts()
    .sort_index()
)

print(combined_counts)

print("\nClass distribution (%):")

print(
    combined_df["labels"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


# ============================================================
# CHECK ORIGINAL COMBINED DATASET
# ============================================================

class_0_count = (
    combined_df["labels"]
    .value_counts()
    .get(0, 0)
)

class_1_count = (
    combined_df["labels"]
    .value_counts()
    .get(1, 0)
)


print("\n====================================")
print("ORIGINAL COMBINED COUNTS")
print("====================================")

print(
    "Class 0:",
    class_0_count
)

print(
    "Class 1:",
    class_1_count
)


# ============================================================
# BALANCE ENTIRE COMBINED DATASET
#
# Keep all 5,804 samples of minority Class 1.
# Randomly undersample Class 0 to 5,804.
# ============================================================

TARGET_PER_CLASS = 5804


if class_0_count < TARGET_PER_CLASS:

    raise ValueError(
        f"Class 0 has only {class_0_count} samples. "
        f"Need at least {TARGET_PER_CLASS}."
    )


if class_1_count < TARGET_PER_CLASS:

    raise ValueError(
        f"Class 1 has only {class_1_count} samples. "
        f"Need at least {TARGET_PER_CLASS}."
    )


# ============================================================
# SEPARATE CLASSES
# ============================================================

class_0 = combined_df[
    combined_df["labels"] == 0
].copy()


class_1 = combined_df[
    combined_df["labels"] == 1
].copy()


# ============================================================
# UNDERSAMPLE CLASS 0
# ============================================================

class_0_balanced = class_0.sample(

    n=TARGET_PER_CLASS,

    random_state=SEED

)


# ============================================================
# KEEP ALL CLASS 1
# ============================================================

class_1_balanced = class_1.sample(

    n=TARGET_PER_CLASS,

    random_state=SEED

)


# ============================================================
# COMBINE BALANCED CLASSES
# ============================================================

balanced_df = pd.concat(

    [
        class_0_balanced,
        class_1_balanced
    ],

    ignore_index=True

)


# ============================================================
# SHUFFLE BALANCED DATASET
# ============================================================

balanced_df = balanced_df.sample(

    frac=1,

    random_state=SEED

).reset_index(drop=True)


# ============================================================
# BALANCED DATASET RESULTS
# ============================================================

print("\n====================================")
print("BALANCED COMBINED DATASET")
print("====================================")

print(
    "Balanced dataset shape:",
    balanced_df.shape
)

print("\nClass distribution:")

balanced_counts = (
    balanced_df["labels"]
    .value_counts()
    .sort_index()
)

print(balanced_counts)

print("\nClass distribution (%):")

print(
    balanced_df["labels"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


# ============================================================
# EXACT BALANCE CHECK
# ============================================================

if (

    balanced_counts.get(0, 0) != 5804

    or

    balanced_counts.get(1, 0) != 5804

):

    raise ValueError(
        "Balanced dataset is NOT 5804 + 5804."
    )


print("\nBALANCED DATASET CHECK PASSED")

print(
    "Class 0 = 5804"
)

print(
    "Class 1 = 5804"
)

print(
    "Total = 11608"
)


# ============================================================
# PREPARE X AND Y
# ============================================================

X = balanced_df[
    "text"
].values


y = balanced_df[
    "labels"
].values


# ============================================================
# 80:20 STRATIFIED TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=SEED,

    stratify=y

)


# ============================================================
# TRAIN / TEST INFORMATION
# ============================================================

print("\n====================================")
print("80:20 STRATIFIED SPLIT")
print("====================================")

print(
    "Training samples:",
    len(X_train)
)

print(
    "Testing samples:",
    len(X_test)
)


# ============================================================
# TRAINING DISTRIBUTION
# ============================================================

train_counts = (
    pd.Series(y_train)
    .value_counts()
    .sort_index()
)


print("\nTraining distribution:")

print(train_counts)


print("\nTraining distribution (%):")

print(
    pd.Series(y_train)
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


# ============================================================
# TESTING DISTRIBUTION
# ============================================================

test_counts = (
    pd.Series(y_test)
    .value_counts()
    .sort_index()
)


print("\nTesting distribution:")

print(test_counts)


print("\nTesting distribution (%):")

print(
    pd.Series(y_test)
    .value_counts(
        normalize=True
    )
    .sort_index()
    .mul(100)
)


# ============================================================
# EXACT SPLIT CHECK
# ============================================================

print("\n====================================")
print("EXACT SPLIT VERIFICATION")
print("====================================")


print(
    "Training Class 0:",
    train_counts.get(0, 0)
)

print(
    "Training Class 1:",
    train_counts.get(1, 0)
)

print(
    "Testing Class 0:",
    test_counts.get(0, 0)
)

print(
    "Testing Class 1:",
    test_counts.get(1, 0)
)


# ============================================================
# HARD CHECK
#
# EXPECTED:
# TRAIN = 4643 + 4643
# TEST  = 1161 + 1161
# ============================================================

if train_counts.get(0, 0) != 4643:

    raise ValueError(
        "Training Class 0 is not 4643."
    )


if train_counts.get(1, 0) != 4643:

    raise ValueError(
        "Training Class 1 is not 4643."
    )


if test_counts.get(0, 0) != 1161:

    raise ValueError(
        "Testing Class 0 is not 1161."
    )


if test_counts.get(1, 0) != 1161:

    raise ValueError(
        "Testing Class 1 is not 1161."
    )


print("\n====================================")
print("SPLIT CHECK PASSED")
print("====================================")

print(
    "Training: 4643 Real + 4643 Fake = 9286"
)

print(
    "Testing: 1161 Real + 1161 Fake = 2322"
)


# ============================================================
# CREATE TRAINING DATAFRAME
# ============================================================

train_df = pd.DataFrame({

    "text": X_train,

    "labels": y_train

})


# ============================================================
# CREATE TESTING DATAFRAME
# ============================================================

test_df = pd.DataFrame({

    "text": X_test,

    "labels": y_test

})


# ============================================================
# CREATE HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(

    train_df,

    preserve_index=False

)


test_dataset = Dataset.from_pandas(

    test_df,

    preserve_index=False

)


# ============================================================
# LOAD mBERT TOKENIZER
# ============================================================

print("\n====================================")
print("LOADING mBERT TOKENIZER")
print("====================================")


tokenizer = BertTokenizer.from_pretrained(

    MODEL_NAME

)


# ============================================================
# TOKENIZATION FUNCTION
# ============================================================

def tokenize_function(examples):

    return tokenizer(

        examples["text"],

        truncation=True,

        padding="max_length",

        max_length=MAX_LENGTH

    )


# ============================================================
# TOKENIZE TRAINING DATA
# ============================================================

print("\nTokenizing training data...")


train_dataset = train_dataset.map(

    tokenize_function,

    batched=True,

    desc="Training tokenization"

)


# ============================================================
# TOKENIZE TESTING DATA
# ============================================================

print("\nTokenizing testing data...")


test_dataset = test_dataset.map(

    tokenize_function,

    batched=True,

    desc="Testing tokenization"

)


# ============================================================
# REMOVE TEXT COLUMN
# ============================================================

train_dataset = train_dataset.remove_columns(

    ["text"]

)


test_dataset = test_dataset.remove_columns(

    ["text"]

)


# ============================================================
# PYTORCH FORMAT
# ============================================================

train_dataset.set_format(

    type="torch",

    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]

)


test_dataset.set_format(

    type="torch",

    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]

)


print("\n====================================")
print("PYTORCH DATASETS")
print("====================================")

print(
    "Training dataset:",
    len(train_dataset)
)

print(
    "Testing dataset:",
    len(test_dataset)
)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()

)


print("\n====================================")
print("DATALOADERS")
print("====================================")

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Testing batches:",
    len(test_loader)
)


# ============================================================
# LOAD mBERT MODEL
# ============================================================

print("\n====================================")
print("LOADING mBERT MODEL")
print("====================================")


model = BertForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2

)


model.to(device)


print("Model loaded successfully.")


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = AdamW(

    model.parameters(),

    lr=LEARNING_RATE

)


# ============================================================
# LOSS FUNCTION
#
# Dataset is balanced.
# Therefore no class weights are needed.
# ============================================================

criterion = CrossEntropyLoss()


# ============================================================
# TRAINING
# ============================================================

print("\n====================================")
print("TRAINING mBERT")
print("COMBINED DATASET - BALANCED")
print("====================================")


training_results = []


for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    correct = 0

    total = 0


    progress_bar = tqdm(

        train_loader,

        total=len(train_loader),

        desc=f"Epoch {epoch + 1}/{EPOCHS}",

        unit="batch"

    )


    for batch in progress_bar:

        # ----------------------------------------------------
        # MOVE DATA TO GPU
        # ----------------------------------------------------

        input_ids = batch[
            "input_ids"
        ].to(

            device,

            non_blocking=True

        )


        attention_mask = batch[
            "attention_mask"
        ].to(

            device,

            non_blocking=True

        )


        labels = batch[
            "labels"
        ].to(

            device,

            non_blocking=True

        )


        # ----------------------------------------------------
        # CLEAR GRADIENTS
        # ----------------------------------------------------

        optimizer.zero_grad()


        # ----------------------------------------------------
        # FORWARD
        # ----------------------------------------------------

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask

        )


        logits = outputs.logits


        # ----------------------------------------------------
        # LOSS
        # ----------------------------------------------------

        loss = criterion(

            logits,

            labels

        )


        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # UPDATE
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # ACCURACY
        # ----------------------------------------------------

        predictions = torch.argmax(

            logits,

            dim=1

        )


        correct += (

            predictions == labels

        ).sum().item()


        total += labels.size(0)


        total_loss += loss.item()


        current_accuracy = (

            correct / total

        )


        progress_bar.set_postfix(

            loss=f"{loss.item():.4f}",

            accuracy=f"{current_accuracy:.4f}"

        )


    # ========================================================
    # EPOCH RESULTS
    # ========================================================

    epoch_loss = (

        total_loss /

        len(train_loader)

    )


    epoch_accuracy = (

        correct /

        total

    )


    training_results.append({

        "epoch": epoch + 1,

        "loss": epoch_loss,

        "accuracy": epoch_accuracy

    })


    print(

        f"\nEpoch {epoch + 1}/{EPOCHS} "

        f"- Loss: {epoch_loss:.4f} "

        f"- Accuracy: {epoch_accuracy:.4f}"

    )


# ============================================================
# TESTING
# ============================================================

print("\n====================================")
print("TESTING mBERT")
print("COMBINED DATASET - BALANCED")
print("====================================")


model.eval()


all_predictions = []

all_labels = []


with torch.no_grad():

    progress_bar = tqdm(

        test_loader,

        total=len(test_loader),

        desc="Testing",

        unit="batch"

    )


    for batch in progress_bar:

        # ----------------------------------------------------
        # MOVE DATA TO GPU
        # ----------------------------------------------------

        input_ids = batch[
            "input_ids"
        ].to(

            device,

            non_blocking=True

        )


        attention_mask = batch[
            "attention_mask"
        ].to(

            device,

            non_blocking=True

        )


        labels = batch[
            "labels"
        ].to(

            device,

            non_blocking=True

        )


        # ----------------------------------------------------
        # PREDICTION
        # ----------------------------------------------------

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask

        )


        logits = outputs.logits


        predictions = torch.argmax(

            logits,

            dim=1

        )


        # ----------------------------------------------------
        # SAVE PREDICTIONS
        # ----------------------------------------------------

        all_predictions.extend(

            predictions
            .cpu()
            .numpy()

        )


        all_labels.extend(

            labels
            .cpu()
            .numpy()

        )


# ============================================================
# FINAL TEST SET VERIFICATION
# ============================================================

print("\n====================================")
print("FINAL TEST SET VERIFICATION")
print("====================================")


actual_test_class_0 = np.sum(

    np.array(all_labels) == 0

)


actual_test_class_1 = np.sum(

    np.array(all_labels) == 1

)


print(
    "Total test samples:",
    len(all_labels)
)

print(
    "Real (0):",
    actual_test_class_0
)

print(
    "Fake (1):",
    actual_test_class_1
)


# ============================================================
# FINAL HARD CHECK
# ============================================================

if actual_test_class_0 != 1161:

    raise ValueError(
        "FINAL TEST ERROR: Real support is not 1161."
    )


if actual_test_class_1 != 1161:

    raise ValueError(
        "FINAL TEST ERROR: Fake support is not 1161."
    )


print("\nFINAL TEST CHECK PASSED")

print(
    "Real (0) = 1161"
)

print(
    "Fake (1) = 1161"
)

print(
    "Total = 2322"
)


# ============================================================
# ACCURACY
# ============================================================

accuracy = accuracy_score(

    all_labels,

    all_predictions

)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

report = classification_report(

    all_labels,

    all_predictions,

    target_names=[

        "Real (0)",

        "Fake (1)"

    ],

    digits=4

)


# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    all_labels,

    all_predictions

)


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n====================================")
print("mBERT EXPERIMENT 6 RESULTS")
print("COMBINED DATASET - BALANCED")
print("====================================")


print(

    f"Test Accuracy: "
    f"{accuracy:.4f}"

)


print(

    f"Test Accuracy: "
    f"{accuracy * 100:.2f}%"

)


print("\nClassification Report:")

print(report)


print("Confusion Matrix:")

print(cm)


# ============================================================
# TRAINING RESULTS
# ============================================================

print("\n====================================")
print("TRAINING RESULTS")
print("====================================")


for result in training_results:

    print(

        f"Epoch {result['epoch']}: "

        f"Loss = {result['loss']:.4f}, "

        f"Accuracy = {result['accuracy']:.4f}"

    )


# ============================================================
# GPU MEMORY
# ============================================================

if torch.cuda.is_available():

    print("\n====================================")
    print("GPU MEMORY")
    print("====================================")


    allocated = (

        torch.cuda.memory_allocated()

        / 1024**3

    )


    reserved = (

        torch.cuda.memory_reserved()

        / 1024**3

    )


    print(

        f"Allocated GPU memory: "
        f"{allocated:.2f} GB"

    )


    print(

        f"Reserved GPU memory: "
        f"{reserved:.2f} GB"

    )

Using device: cuda
GPU: Tesla T4

DATASET A
Dataset A shape: (14564, 2)

Class distribution:
Is Fake
0    11662
1     2902
Name: count, dtype: int64

Class distribution (%):
Is Fake
0    80.074155
1    19.925845
Name: proportion, dtype: float64

DATASET B
Dataset B shape: (5226, 2)

Class distribution:
Authenticity
0    2324
1    2902
Name: count, dtype: int64

Class distribution (%):
Authenticity
0    44.469958
1    55.530042
Name: proportion, dtype: float64

COMBINED DATASET - IMBALANCED
Combined dataset shape: (19790, 2)

Class distribution:
labels
0    13986
1     5804
Name: count, dtype: int64

Class distribution (%):
labels
0    70.672057
1    29.327943
Name: proportion, dtype: float64

ORIGINAL COMBINED COUNTS
Class 0: 13986
Class 1: 5804

BALANCED COMBINED DATASET
Balanced dataset shape: (11608, 2)

Class distribution:
labels
0    5804
1    5804
Name: count, dtype: int64

Class distribution (%):
labels
0    50.0
1    50.0
Name: proportion, dtype: float64

BALANCED DATASET CHECK

Training tokenization:   0%|          | 0/9286 [00:00<?, ? examples/s]


Tokenizing testing data...


Testing tokenization:   0%|          | 0/2322 [00:00<?, ? examples/s]


PYTORCH DATASETS
Training dataset: 9286
Testing dataset: 2322

DATALOADERS
Training batches: 581
Testing batches: 146

LOADING mBERT MODEL


...


Epoch 2/3 - Loss: 0.0530 - Accuracy: 0.9845


Epoch 3/3:   0%|          | 0/581 [00:00<?, ?batch/s]


Epoch 3/3 - Loss: 0.0329 - Accuracy: 0.9914

TESTING mBERT
COMBINED DATASET - BALANCED


Testing:   0%|          | 0/146 [00:00<?, ?batch/s]


FINAL TEST SET VERIFICATION
Total test samples: 2322
Real (0): 1161
Fake (1): 1161

FINAL TEST CHECK PASSED
Real (0) = 1161
Fake (1) = 1161
Total = 2322

mBERT EXPERIMENT 6 RESULTS
COMBINED DATASET - BALANCED
Test Accuracy: 0.9858
Test Accuracy: 98.58%

Classification Report:
              precision    recall  f1-score   support

    Real (0)     0.9829    0.9888    0.9858      1161
    Fake (1)     0.9887    0.9828    0.9857      1161

    accuracy                         0.9858      2322
   macro avg     0.9858    0.9858    0.9858      2322
weighted avg     0.9858    0.9858    0.9858      2322

Confusion Matrix:
[[1148   13]
 [  20 1141]]

TRAINING RESULTS
Epoch 1: Loss = 0.1307, Accuracy = 0.9543
Epoch 2: Loss = 0.0530, Accuracy = 0.9845
Epoch 3: Loss = 0.0329, Accuracy = 0.9914

GPU MEMORY
Allocated GPU memory: 5.32 GB
Reserved GPU memory: 6.95 GB
